# Практическая работа №9 — SVM (вариант A)

**Цель:** сравнить линейное и RBF-ядра на двумерном наборе make_moons и исследовать влияние C.

**Параметры:** 320 объектов, шум 0.22, random_state=42, train/test = 75%/25%. Масштабирование StandardScaler применяется внутри Pipeline и обучается только на тренировочных объектах.

In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "svm_variant_a.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "svm_variant_a.py").exists(), "Запустите ноутбук из папки проекта"
sys.path.insert(0, str(ROOT))
from src.svm_variant_a import (
    load_data, build_models, fit_and_score, save_comparison
)

## 1. Датасет и разделение

make_moons создаёт два изогнутых класса, поэтому линейная граница не может идеально их разделить. Сначала выделяем тестовую часть и только затем масштабируем признаки в процессе fit Pipeline.

In [ ]:
X_train, X_test, y_train, y_test = load_data()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Обучение и метрики

В каждой модели установлен Pipeline: StandardScaler → SVC. Linear и RBF сравниваются при C=1. Для исследования влияния C обучаются три RBF-модели с C=0.05, 1 и 100 (gamma=scale).

In [ ]:
models = build_models()
for key, model in models.items():
    print(key, ":", model)
metrics = fit_and_score(models, X_train, X_test, y_train, y_test)
rows = ["| Модель | Train accuracy | Test accuracy | F1 на тесте | Опорных векторов |",
        "|---|---:|---:|---:|---:|"]
for key, r in metrics.items():
    rows.append(f"| {key} | {r['train_accuracy']:.4f} | {r['test_accuracy']:.4f} | {r['test_f1']:.4f} | {r['n_support_vectors']} |")
display(Markdown("\n".join(rows)))

## 3. Визуализация границы решения

Цвет точек означает принадлежность к классу. Чёрная линия соответствует decision_function=0, а обведённые точки — опорные векторы. Оси показаны в исходной шкале; перед предсказанием каждая модель преобразует данные через свой StandardScaler.

In [ ]:
out = ROOT / "results"
save_comparison(
    models, metrics, X_train, y_train,
    [("linear_C1", "Linear, C=1"), ("rbf_C1", "RBF, C=1")],
    "Сравнение ядер SVM", out / "kernels_comparison.png"
)
display(Image(filename=str(out / "kernels_comparison.png"), width=1050))

## 4. Исследование параметра C

При меньшем C модель охотнее допускает ошибки ради более простой границы. При большом C ошибки штрафуются сильнее, из-за чего граница может стать сложнее. Слишком большое C повышает риск переобучения.

In [ ]:
save_comparison(
    models, metrics, X_train, y_train,
    [("rbf_C005", "RBF, C=0.05"),
     ("rbf_C1", "RBF, C=1"),
     ("rbf_C100", "RBF, C=100")],
    "Влияние параметра C", out / "c_effect.png"
)
display(Image(filename=str(out / "c_effect.png"), width=1400))

In [ ]:
payload = {"dataset": "make_moons", "n_samples": 320, "noise": 0.22,
           "random_state": 42, "train_size": len(y_train),
           "test_size": len(y_test), "models": metrics}
(out / "metrics.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)
print("Результаты и графики сохранены в", out)

## 5. Письменные ответы

**Какое ядро лучше?** RBF при C=1 даёт test accuracy **0.9250** против **0.8625** у Linear C=1. Причина: классы make_moons образуют две дуги, и нелинейное ядро лучше описывает границу между ними.

**Как C влияет на результат?** При C=0.05 RBF точность равна **0.8750**, опорных векторов **160**. При C=100 точность равна **0.9750**, опорных векторов **44**. Повышение C привело к более сложной границе и меньшему числу опорных векторов на этом конкретном разбиении.

**Вывод:** RBF лучше линейного ядра на выбранных нелинейных данных. Это оценка на одном фиксированном train/test split; выигрыш при C=100 не гарантируется на другой выборке. Для подбора параметров в реальной задаче используйте кросс-валидацию.